In [ ]:
from typing import Dict, List, Optional
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_community.llms import Ollama
import json
import re

class WorkflowState:
    def __init__(self):
        self.current_phase = "Personal Information"
        self.phases = {
            "Personal Information": {
                "completed": False,
                "data": {
                    'full_name': None,
                    'company_name': None,
                    'company_address': None,
                    'phone_number': None,
                    'email': None
                }
            },
            "Container Check": {
                "completed": False,
                "data": {
                    'number_of_containers': None
                }
            },
            "Container Details": {  # More than 3 containers start
                "completed": False,
                "data": {
                    'size_of_container': None,
                    'empty_or_loaded': None,
                    'pickup_address': None
                    
                },
                "check": 0                 
            },
            
            "Delivery Detail": { 
                "completed": False,
                "data": {
                    'delivery_address': None
                    
                },
                "check": 0                 
            },
            
            
            
            
            # More than 3 containers end
            
            
            
            "less than 3": { # Less than 3 containers start
                "completed": False,
                "data": {
                    'number_of_containers': None
                }
            },
            # Less than 3 containers end
            "Workflow complete": {
                "completed": False,
                "data": {}
            }
        }
        self.integrations = {
            'sap_connection': False,
            'ms_dynamics_sync': False
        }

    def get_current_requirements(self, current_phase) -> List[str]:
        return [k for k, v in self.phases[current_phase]["data"].items() if v is None]

    def update_phase_data(self, new_data: Dict):
        current_data = self.phases[self.current_phase]["data"]
            # If data is a list (e.g., multiple entries), combine values for each key
        if isinstance(new_data, list):
            aggregated_data = {key: [] for key in current_data.keys()}

            for entry in new_data:
                for key, value in entry.items():
                    if key in aggregated_data and value is not None:
                        aggregated_data[key].append(value)

            # Flatten lists to single values if they only contain one element
            for key, value_list in aggregated_data.items():
                if value_list:
                    if len(value_list) == 1:
                        current_data[key] = value_list[0]
                    else:
                        current_data[key] = value_list

        else:
            # Single dictionary case (existing behavior)
            for key, value in new_data.items():
                if key in current_data and current_data[key] is None and value is not None:
                    current_data[key] = value

        # Check if phase complete (all fields non-None)
        if all(v is not None for v in current_data.values()):
            self.phases[self.current_phase]["completed"] = True
            self._transition_to_next_phase()

    def _transition_to_next_phase(self):
        transitions = {
            "Personal Information": "Container Check",
            "Container Check": self._determine_container_phase,
            "Container Details": "Delivery Detail",
            "Delivery Detail": "Workflow complete",
            
            
            "less than 3": "Workflow complete"
        }

        next_phase = transitions.get(self.current_phase)
        if callable(next_phase):
            next_phase = next_phase()

        if next_phase:
            self.current_phase = next_phase

    def _determine_container_phase(self):
        number_of_containers = self.phases["Container Check"]["data"]["number_of_containers"]
        if isinstance(number_of_containers, list):
            number_of_containers = [int(x) for x in number_of_containers]
            number_of_containers = sum(number_of_containers)
        
        elif isinstance(number_of_containers, str):
            number_of_containers = int(number_of_containers)
            
        if number_of_containers >= 3:
            return "Container Details"
        else:
            return "less than 3"

class WorkflowManager:
    def __init__(self):
        self.llm = Ollama(model="deepseek-r1:8b", base_url="http://localhost:11434")
        self.state = WorkflowState()
        
        self.phase_prompts = {
            "Personal Information": PromptTemplate(
                input_variables=["message"],
                template=
                """Extract following entities from text. Return as JSON. The entries that are missing should be null in the json.
                Entities that you need to extract are: full_name, company_name, company_address, phone_number and email.
                
                Input: {message}
                JSON:"""
            ),
            "Container Check": PromptTemplate(
                input_variables=["message"],
                template=
                """Extract following entities from text. Return as JSON. The entries that are missing should be null in the json.
                Entities that you need to extract are: number_of_containers.
                
                Input: {message}
                JSON:"""
            ),
            "Container Details": PromptTemplate(
                input_variables=["message"],
                template=
                """Extract following entities from text. Return as JSON. The entries that are missing should be null in the json.
                Entities that you need to extract are: size_of_container, empty_or_loaded, pickup_address.
                
                Input: {message}
                JSON:"""
            ),
            "Delivery Detail": PromptTemplate(
                input_variables=["message"],
                template=
                """Extract following entities from text. Return as JSON. The entries that are missing should be null in the json.
                Entities that you need to extract are: delivery_address.
                
                Input: {message}
                JSON:"""
            ),
            
            
            
            
            
            "less than 3": PromptTemplate(
                input_variables=["message"],
                template=
                """Extract following entities from text. Return as JSON. The entries that are missing should be null in the json.
                Entities that you need to extract are: number_of_containers.
                
                Input: {message}
                JSON:"""
            ),
        }
        
        self.response_templates = {
            "Personal Information": "Please provide {missing} for the Personal Information phase",
            "Container Check": "Need clarification on {missing} for Container Check phase",
            "Container Details": "The current phase is Container Details",
            "Delivery Detail": "The current phase is Delivery Detail",
            "less than 3": "The current phase is less than 3",
        }
        self.user_promt = {
            "Personal Information": "Please provide your personal information including full name, company name, address, phone number, and email.",
            "Container Check": "Please provide the container number for the shipment",
            "Container Details": """Thank you! Since you're transporting multiple containers, one of our team members will contact you to assist with scheduling. 
                                    Before submitting, could you provide details for the first container—specifically,
                                    its size (20ft, 40ft, or high cube) and whether its empty or loaded? Also prove the Pickup address for it.""",
            "Delivery Detail": """Great, thank you! Could you provide the delivery location? like before this can be an address or coordinates 
                                            if it's in an area without a defined address.""",
            "less than 3": "The current phase is less than 3",
            "Workflow complete": "Workflow complete"
        }

    def process_input(self, user_input: str) -> str:
        # Get current phase requirements
        current_phase = self.state.current_phase
        
        # Process phase-specific data
        extraction_chain = LLMChain(
            llm=self.llm,
            prompt=self.phase_prompts.get(current_phase, self.phase_prompts["Personal Information"])
        )
        
        try:
            respon = extraction_chain.run(message=user_input)
            print("Response", clean_deepseek_response(respon))
            extracted_data = json.loads(clean_deepseek_response(respon))
            self.state.update_phase_data(extracted_data)
        except json.JSONDecodeError as e:
            return f"Error processing input. Please try again.{e}"

        # Generate phase-appropriate response
        missing = self.state.get_current_requirements(current_phase)
        
        if missing:
            response_template = self.response_templates[current_phase]
            # print("Promt: ",PromptTemplate(input_variables=["missing"],template=response_template.format(missing=", ".join(missing))))
            if self.state.phases["Personal Information"]["data"]["full_name"] is not None:
                user_name = self.state.phases["Personal Information"]["data"]["full_name"]
            else:
                user_name = "User"
            missing_llm = LLMChain(
                llm=self.llm,
                prompt=PromptTemplate(
                    input_variables=["message", "user_name"],
                    template=
                    """You are a bot that needs to ask the user for the missing information. Start by addressing the user directly as {user_name}.This is for shipping purposes.
                    Let the user know about the following missing details that were not read or entered: {message}.
                    Politely ask the user to provide the required information. Do not include any extra explanations or formalities—just ask for the details. """
                )
            )
            missing_response = clean_deepseek_response(missing_llm.run(message=missing,user_name=user_name))
            print("Response for missing", missing_response)
            return missing_response
        
        return f"Phase {current_phase} completed. Moving to {self.state.current_phase}"

def clean_deepseek_response(response: str) -> str:
    pattern = r"<think>(.*?)</think>"
    cleaned_response = re.sub(pattern, "", response, flags=re.DOTALL).strip()
    json_pattern = re.sub(r"^```json|\n```$", "", cleaned_response.strip(), flags=re.MULTILINE).strip()
    return json_pattern

# Example Usage
manager = WorkflowManager()

test_inputs = [
    "HI I am John Doe from ABC Inc. My address is 123 Main St, Dallas, TX 75201. Phone: 555-123-4567, Email: jhondoe@gmail.com",
    "We have 4 containers to  ship",
    "The container is 40ft and loaded. Pickup address is 456 Elm St, Dallas, TX 75202"
]

# for input in test_inputs:
#     print(f"Input: {input}")
#     print(f"System: {manager.process_input(input)}\n")
#     print(f"Current Phase: {manager.state.current_phase}")
#     print(f"Phase Data: {manager.state.phases}\n")

while True:
    print(f"System: {manager.user_promt[manager.state.current_phase]}")
    user_input = input("You: ")
    
    if user_input.lower() == "quit":
        print("Bot: Goodbye!")
        break
    response = manager.process_input(user_input)
    
    print(f"System: {response}\n")
    print(f"Current Phase: {manager.state.current_phase}")
    print(f"Phase Data: {manager.state.phases}\n")
    if manager.state.current_phase == "Workflow complete":
        break

System: Please provide your personal information including full name, company name, address, phone number, and email.
Response {
  "full_name": "John Doe",
  "company_name": "ABC Inc.",
  "company_address": "123 Main St, Dallas, TX 75201",
  "phone_number": "555-123-4567",
  "email": "jhondoe@gmail.com"
}
System: Phase Personal Information completed. Moving to Container Check

Current Phase: Container Check
Phase Data: {'Personal Information': {'completed': True, 'data': {'full_name': 'John Doe', 'company_name': 'ABC Inc.', 'company_address': '123 Main St, Dallas, TX 75201', 'phone_number': '555-123-4567', 'email': 'jhondoe@gmail.com'}}, 'Container Check': {'completed': False, 'data': {'number_of_containers': None}}, 'Container Details': {'completed': False, 'data': {'size_of_container': None, 'empty_or_loaded': None, 'pickup_address': None}, 'check': 0}, 'Delivery Detail': {'completed': False, 'data': {'delivery_address': None}, 'check': 0}, 'less than 3': {'completed': False, 'data':

In [38]:
import json
import re
from typing import Optional, TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_community.llms import Ollama

def clean_deepseek_response(response: str) -> str:
    pattern = r"<think>(.*?)</think>"
    cleaned_response = re.sub(pattern, "", response, flags=re.DOTALL).strip()
    # cleaned_response = re.sub(r"^\"|\n\"$", "", cleaned_response.strip(), flags=re.MULTILINE).strip()
    return cleaned_response

# Define the state model for our shipping workflow.
class ShippingWorkflowState(TypedDict):
    current_phase: str
    full_name: Optional[str]
    company_name: Optional[str]
    company_address: Optional[str]
    phone_number: Optional[str]
    email: Optional[str]
    number_of_containers: Optional[str]
    size_of_container: Optional[str]
    empty_or_loaded: Optional[str]
    pickup_address: Optional[str]
    delivery_address: Optional[str]
    response: Optional[str]

# Initialize the LLM (using your local Ollama instance).
llm = Ollama(model="deepseek-r1:8b", base_url="http://localhost:11434")

# A prompt to ask the user for missing information in a friendly, conversational style.
User_info_prompt = PromptTemplate(
    input_variables=["message"],
    template=(
        "You are a friendly assistant. Please generate a message to ask the user for more information. "
        "Return only the final message (do not include any extra commentary or explanation). "
        "The message should be: 'Hi there, I noticed we still need some information: {message}. Could you please provide that for me?'"
    )
)

# Extraction prompts for each phase.
phase_prompts = {
    "Personal Information": PromptTemplate(
        input_variables=["message"],
        template=(
            "Extract the following entities as JSON (missing fields should be null):\n"
            "full_name, company_name, company_address, phone_number, email.\n\n"
            "Input: {message}\nJSON:"
        )
    ),
    "Container Check": PromptTemplate(
        input_variables=["message"],
        template=(
            "Extract the following entity as JSON (missing fields should be null):\n"
            "number_of_containers.\n\n"
            "Input: {message}\nJSON:"
        )
    ),
    "Container Details": PromptTemplate(
        input_variables=["message"],
        template=(
            "Extract the following entities as JSON (missing fields should be null):\n"
            "size_of_container, empty_or_loaded, pickup_address.\n\n"
            "Input: {message}\nJSON:"
        )
    ),
    "Delivery Detail": PromptTemplate(
        input_variables=["message"],
        template=(
            "Extract the following entity as JSON (missing fields should be null):\n"
            "delivery_address.\n\n"
            "Input: {message}\nJSON:"
        )
    ),
    "less than 3": PromptTemplate(
        input_variables=["message"],
        template=(
            "Extract the following entity as JSON (missing fields should be null):\n"
            "number_of_containers.\n\n"
            "Input: {message}\nJSON:"
        )
    ),
}

# --- Node Functions ---
# Each node now responds in a human-like way and, when all required info is present, updates the current phase.

def personal_info_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    missing = [field for field in ["full_name", "company_name", "company_address", "phone_number", "email"]
               if not state.get(field)]
    if missing:
        # Ask the user for missing info in a friendly manner.
        friendly_message = LLMChain(llm=llm, prompt=User_info_prompt).run(message="".join(missing))
        state["response"] = clean_deepseek_response(friendly_message)
        state["current_phase"] = "Personal Information"
    else:
        state["response"] = "Great! I have all your personal information. Let's move on to the next step."
        state["current_phase"] = "Container Check"
    return state

def container_check_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    if not state.get("number_of_containers"):
        
        # state["response"] = "I didn't catch how many containers you have. Could you please tell me the number of containers for your shipment?"
        
        friendly_message = LLMChain(llm=llm, prompt=User_info_prompt).run(message="".join("number_of_containers"))
        state["response"] = clean_deepseek_response(friendly_message)
        
        state["current_phase"] = "Container Check"
    else:
        try:
            count = int(state["number_of_containers"])
        except Exception:
            count = 0
        state["response"] = f"Thanks! You have {state['number_of_containers']} container(s)."
        state["current_phase"] = "Container Details" if count >= 3 else "less than 3"
    return state

def container_details_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    missing = [field for field in ["size_of_container", "empty_or_loaded", "pickup_address"]
               if not state.get(field)]
    if missing:
        # state["response"] = f"I need a few more details about your containers: {', '.join(missing)}. Could you please provide those?"
        
        friendly_message = LLMChain(llm=llm, prompt=User_info_prompt).run(message="".join(missing))
        state["response"] = clean_deepseek_response(friendly_message)
        
        state["current_phase"] = "Container Details"
    else:
        state["response"] = "Thanks for providing the container details. Now, please let me know the delivery address."
        state["current_phase"] = "Delivery Detail"
    return state

def delivery_detail_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    if not state.get("delivery_address"):
        # state["response"] = "Could you please tell me the delivery address?"
        
        friendly_message = LLMChain(llm=llm, prompt=User_info_prompt).run(message="".join("delivery_address"))
        state["response"] = clean_deepseek_response(friendly_message)
        
        state["current_phase"] = "Delivery Detail"
    else:
        state["response"] = "Perfect! All delivery details are set."
        state["current_phase"] = "Workflow complete"
    return state

def less_than_3_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    state["response"] = "Alright, since you have less than 3 containers, we can proceed without extra container details."
    state["current_phase"] = "Workflow complete"
    return state

def workflow_complete_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    state["response"] = "Everything is complete. Thank you for providing all the required information!"
    return state

def wait_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    state["response"] = state.get("response", "I'm waiting for your input...")
    return state

# --- Build the langGraph state graph ---
workflow = StateGraph(ShippingWorkflowState)

workflow.add_node("Personal Information", personal_info_node)
workflow.add_node("Container Check", container_check_node)
workflow.add_node("Container Details", container_details_node)
workflow.add_node("Delivery Detail", delivery_detail_node)
workflow.add_node("less than 3", less_than_3_node)
workflow.add_node("Workflow complete", workflow_complete_node)
workflow.add_node("Wait", wait_node)

# Add an entrypoint.
workflow.add_edge(START, "Personal Information")

# Use conditional edges to route based on missing information.
workflow.add_conditional_edges(
    "Personal Information",
    lambda state: "Wait" if any(not state.get(field) for field in ["full_name", "company_name", "company_address", "phone_number", "email"])
                  else "Container Check",
    {"Wait": "Wait", "Container Check": "Container Check"}
)
workflow.add_conditional_edges(
    "Container Check",
    lambda state: "Wait" if not state.get("number_of_containers")
                  else ("Container Details" if int(state["number_of_containers"]) >= 3 else "less than 3"),
    {"Wait": "Wait", "Container Details": "Container Details", "less than 3": "less than 3"}
)
workflow.add_conditional_edges(
    "Container Details",
    lambda state: "Wait" if any(not state.get(field) for field in ["size_of_container", "empty_or_loaded", "pickup_address"])
                  else "Delivery Detail",
    {"Wait": "Wait", "Delivery Detail": "Delivery Detail"}
)
workflow.add_conditional_edges(
    "Delivery Detail",
    lambda state: "Wait" if not state.get("delivery_address") else "Workflow complete",
    {"Wait": "Wait", "Workflow complete": "Workflow complete"}
)
workflow.add_edge("less than 3", "Workflow complete")
workflow.add_edge("Workflow complete", END)

app = workflow.compile()

# --- Processing User Input ---
def process_phase_input(state: ShippingWorkflowState, user_input: str) -> ShippingWorkflowState:
    current_phase = state["current_phase"]
    prompt_template = phase_prompts.get(current_phase)
    if prompt_template:
        extraction_chain = LLMChain(llm=llm, prompt=prompt_template)
        try:
            response = extraction_chain.run(message=user_input)
            cleaned = re.sub(r"<think>(.*?)</think>", "", response, flags=re.DOTALL).strip()
            match = re.search(r"```json(.*?)```", cleaned, re.DOTALL)
            if match:
                cleaned = match.group(1).strip()
            extracted_data = json.loads(cleaned)
            for key, value in extracted_data.items():
                if value is not None and not state.get(key):
                    state[key] = value
        except Exception as e:
            state["response"] = f"Error processing input: {e}"
    return state

# --- Main Conversation Loop ---
def main():
    state: ShippingWorkflowState = {
        "current_phase": "Personal Information",
        "full_name": None,
        "company_name": None,
        "company_address": None,
        "phone_number": None,
        "email": None,
        "number_of_containers": None,
        "size_of_container": None,
        "empty_or_loaded": None,
        "pickup_address": None,
        "delivery_address": None,
        "response": ""
    }
    
    print("Welcome! I'm here to help you with your shipment. Let's get started. (Type 'q' to exit.)\n")
    
    while state["current_phase"] != "Workflow complete":
        state = app.invoke(state)
        print(f"System: {state['response']}")
        
        if state["current_phase"] in ["Personal Information", "Container Check", "Container Details", "Delivery Detail"]:
            user_input = input("You: ")
            if user_input.lower() == "q":
                print("Goodbye!")
                return
            state = process_phase_input(state, user_input)
        else:
            input("Press Enter to continue...")
    
    state = app.invoke(state)
    print(f"System: {state['response']}")
    
if __name__ == "__main__":
    main()


Welcome! I'm here to help you with your shipment. Let's get started. (Type 'q' to exit.)

System: Hi there, I noticed we still need some information: full_namecompany_namecompany_addressphone_numberemail. Could you please provide that for me?
System: Hi there, I noticed we still need some information: email. Could you please provide that for me?
System: Hi there, I noticed we still need some information: number_of_containers. Could you please provide that for me?
System: Everything is complete. Thank you for providing all the required information!
System: Everything is complete. Thank you for providing all the required information!


In [40]:
state

{'full_name': None,
 'company_name': 'ABC Inc.',
 'company_address': '123 Main St, Dallas, TX 75201',
 'phone_number': '555-123-4567',
 'email': 'johndoe@gmail.com',
 'number_of_containers': '4',
 'size_of_container': '40ft',
 'empty_or_loaded': 'loaded',
 'pickup_address': '456 Elm St, Dallas, TX 75202',
 'delivery_address': '789 Oak St, Dallas, TX 75203',
 'response': None}

In [ ]:

test_inputs = [
    "HI I am John Doe from ABC Inc. My address is 123 Main St, Dallas, TX 75201. Phone: 555-123-4567, Email: jhondoe@gmail.com",
    "We have 4 containers to  ship",
    "The container is 40ft and loaded. Pickup address is 456 Elm St, Dallas, TX 75202",
    "The delivery address is 456 Elm St, Dallas, TX 75202"
]

In [ ]:
re.search(r"```json(.*?)```", response.text, re.DOTALL).group(1).strip()